# 📱 Sistem Prediksi Rentang Harga dan Rekomendasi Smartphone
## Menggunakan Random Forest Classifier

**Mata Kuliah:** Proyek Data Mining (ST167) — Universitas Amikom Yogyakarta  
**Algoritma:** Random Forest Classifier  
**Dataset:** Mobile Price Classification (Kaggle)

---
### Alur Notebook
1. 📦 Import Library
2. 📂 Load Dataset
3. 🔍 Exploratory Data Analysis (EDA)
4. ⚙️ Preprocessing
5. 🌲 Modelling — Random Forest
6. 📊 Evaluasi Model
7. 💾 Simpan Model (.pkl)

---
## 1. 📦 Import Library

In [ ]:
# ── Core
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Visualisasi
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='Set2')

# ── Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

# ── Simpan model
import joblib

print('✅ Semua library berhasil diimport!')

---
## 2. 📂 Load Dataset

In [ ]:
# ── Upload file jika pakai Colab
# from google.colab import files
# uploaded = files.upload()   # upload train.csv

# ── Load CSV  (sesuaikan path jika perlu)
df = pd.read_csv('train.csv')

print(f'Shape dataset : {df.shape}')
print(f'Jumlah baris  : {df.shape[0]:,}')
print(f'Jumlah kolom  : {df.shape[1]}')
df.head()

In [ ]:
# ── Info tipe data
df.info()

In [ ]:
# ── Statistik deskriptif
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean','std','min','max'])

---
## 3. 🔍 Exploratory Data Analysis (EDA)

### 3.1 Cek Missing Value & Duplikat

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else '✅ Tidak ada missing value!')

print(f'\n=== Duplikasi ===')
dup = df.duplicated().sum()
print(f'✅ Tidak ada duplikat!' if dup == 0 else f'⚠️ Ditemukan {dup} baris duplikat.')

### 3.2 Distribusi Target (price_range)

In [ ]:
label_map = {0: 'Murah\n(0)', 1: 'Menengah\n(1)', 2: 'Mahal\n(2)', 3: 'Premium\n(3)'}
harga_map = {0: 'Rp 500rb–2jt', 1: 'Rp 2–4jt', 2: 'Rp 4–7jt', 3: 'Rp 7jt+'}
colors    = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0']

counts = df['price_range'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar([label_map[i] for i in counts.index], counts.values, color=colors)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', fontweight='bold')
axes[0].set_title('Distribusi Kelas Harga (Bar Chart)', fontweight='bold')
axes[0].set_ylabel('Jumlah HP')
axes[0].set_ylim(0, counts.max() * 1.15)

# Pie chart
axes[1].pie(counts.values,
            labels=[f"{label_map[i]}\n{harga_map[i]}" for i in counts.index],
            colors=colors, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor':'white', 'linewidth':1.5})
axes[1].set_title('Distribusi Kelas Harga (Pie Chart)', fontweight='bold')

plt.suptitle('Distribusi Target: price_range', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_distribusi_target.png', bbox_inches='tight')
plt.show()
print('\n', counts.rename('Jumlah').to_frame())

### 3.3 Distribusi Fitur Numerik

In [ ]:
num_features = df.select_dtypes(include=np.number).columns.drop('price_range').tolist()

n_cols = 4
n_rows = (len(num_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].hist(df[col], bins=30, color='#2196F3', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Nilai')
    axes[i].set_ylabel('Frekuensi')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi Setiap Fitur Numerik', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_distribusi_fitur.png', bbox_inches='tight')
plt.show()

### 3.4 Boxplot: Fitur Penting vs price_range

In [ ]:
key_features = ['ram', 'battery_power', 'px_height', 'px_width', 'int_memory', 'pc']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    data_by_class = [df[df['price_range'] == c][feat].values for c in range(4)]
    bp = axes[i].boxplot(data_by_class, patch_artist=True,
                         medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    axes[i].set_title(f'{feat} vs price_range', fontweight='bold')
    axes[i].set_xticklabels(['Murah', 'Menengah', 'Mahal', 'Premium'])
    axes[i].set_xlabel('Kelas Harga')
    axes[i].set_ylabel(feat)

plt.suptitle('Boxplot Fitur Kunci vs Kelas Harga', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_boxplot_fitur.png', bbox_inches='tight')
plt.show()

### 3.5 Heatmap Korelasi

In [ ]:
plt.figure(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 7})
plt.title('Heatmap Korelasi Antar Fitur', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_heatmap_korelasi.png', bbox_inches='tight')
plt.show()

# Korelasi terhadap target
print('\n=== Korelasi Fitur terhadap price_range (sorted) ===')
print(corr['price_range'].drop('price_range').abs().sort_values(ascending=False).to_string())

---
## 4. ⚙️ Preprocessing

In [ ]:
# ── Pisahkan fitur & target
X = df.drop('price_range', axis=1)
y = df['price_range']

print(f'Shape X : {X.shape}')
print(f'Shape y : {y.shape}')
print(f'Distribusi kelas:\n{y.value_counts().sort_index()}')

In [ ]:
# ── Train-Test Split  (80:20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set : {X_train.shape[0]:,} baris')
print(f'Test set  : {X_test.shape[0]:,} baris')
print()
print('Distribusi kelas TRAIN:')
print(y_train.value_counts().sort_index())
print('\nDistribusi kelas TEST:')
print(y_test.value_counts().sort_index())

> **Catatan:** Dataset ini sudah berupa angka semua (tidak ada kolom kategorikal string), sehingga tidak diperlukan Label Encoding atau One-Hot Encoding tambahan. Random Forest juga tidak sensitif terhadap perbedaan skala fitur, sehingga **normalisasi/standarisasi tidak wajib** dilakukan.

---
## 5. 🌲 Modelling — Random Forest Classifier

In [ ]:
# ── Inisialisasi & Training
rf_model = RandomForestClassifier(
    n_estimators    = 200,      # jumlah pohon
    max_depth       = None,     # kedalaman tak terbatas
    min_samples_split = 2,
    min_samples_leaf  = 1,
    max_features    = 'sqrt',   # jumlah fitur per split
    class_weight    = 'balanced',
    random_state    = 42,
    n_jobs          = -1        # pakai semua core
)

rf_model.fit(X_train, y_train)
print('✅ Model Random Forest berhasil ditraining!')
print(f'   Jumlah pohon     : {rf_model.n_estimators}')
print(f'   Jumlah fitur     : {rf_model.n_features_in_}')

In [ ]:
# ── Cross-Validation (5-Fold Stratified)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

print('=== Cross-Validation (5-Fold) ===')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f} ({score*100:.2f}%)')
print(f'\n  Mean  : {cv_scores.mean():.4f} ({cv_scores.mean()*100:.2f}%)')
print(f'  Std   : {cv_scores.std():.4f}')

---
## 6. 📊 Evaluasi Model

### 6.1 Accuracy, Precision, Recall, F1-Score

In [ ]:
y_pred      = rf_model.predict(X_test)
y_pred_prob = rf_model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'🎯 Accuracy : {acc:.4f} ({acc*100:.2f}%)\n')

class_names = ['Murah (0)', 'Menengah (1)', 'Mahal (2)', 'Premium (3)']
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=class_names))

### 6.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix (Count)', fontweight='bold')
axes[0].set_xticklabels(class_names, rotation=20, ha='right')

# Normalized
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm, display_labels=class_names).plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')
axes[1].set_xticklabels(class_names, rotation=20, ha='right')

plt.suptitle('Confusion Matrix — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_confusion_matrix.png', bbox_inches='tight')
plt.show()

### 6.3 ROC-AUC Curve (One-vs-Rest)

In [ ]:
n_classes = 4
y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))

# ROC per kelas
fpr, tpr, roc_auc = {}, {}, {}
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Micro-average
fpr['micro'], tpr['micro'], _ = roc_curve(y_test_bin.ravel(), y_pred_prob.ravel())
roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])

# Plot
plt.figure(figsize=(8, 6))
plt.plot(fpr['micro'], tpr['micro'],
         label=f'Micro-avg (AUC = {roc_auc["micro"]:.4f})',
         color='black', linestyle='--', linewidth=2)

class_labels = ['Murah', 'Menengah', 'Mahal', 'Premium']
for i, color in enumerate(colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'{class_labels[i]} (AUC = {roc_auc[i]:.4f})')

plt.plot([0,1],[0,1], 'k:', linewidth=1)
plt.xlim([0, 1]); plt.ylim([0, 1.02])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Random Forest (One-vs-Rest)', fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('plot_roc_auc.png', bbox_inches='tight')
plt.show()

print('=== ROC-AUC Score ===')
for i, label in enumerate(class_labels):
    print(f'  {label:10s}: {roc_auc[i]:.4f}')
print(f'  Micro-avg : {roc_auc["micro"]:.4f}')

# Macro ROC-AUC
macro_roc = roc_auc_score(y_test_bin, y_pred_prob, average='macro', multi_class='ovr')
print(f'  Macro-avg : {macro_roc:.4f}')

### 6.4 Feature Importance

In [ ]:
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns)\
             .sort_values(ascending=True)

plt.figure(figsize=(8, 7))
bars = plt.barh(feat_imp.index, feat_imp.values,
                color=plt.cm.RdYlGn(feat_imp.values / feat_imp.values.max()))
for bar, val in zip(bars, feat_imp.values):
    plt.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=8)
plt.xlabel('Importance Score')
plt.title('Feature Importance — Random Forest', fontweight='bold')
plt.tight_layout()
plt.savefig('plot_feature_importance.png', bbox_inches='tight')
plt.show()

print('\n=== Top 5 Fitur Paling Berpengaruh ===')
print(feat_imp.sort_values(ascending=False).head())

### 6.5 Ringkasan Evaluasi

In [ ]:
from sklearn.metrics import f1_score

f1_macro  = f1_score(y_test, y_pred, average='macro')
f1_weight = f1_score(y_test, y_pred, average='weighted')

print('╔══════════════════════════════════════════╗')
print('║      RINGKASAN EVALUASI MODEL            ║')
print('╠══════════════════════════════════════════╣')
print(f'║  Accuracy              : {acc:.4f} ({acc*100:.2f}%)  ║')
print(f'║  F1-Score (Macro)      : {f1_macro:.4f}            ║')
print(f'║  F1-Score (Weighted)   : {f1_weight:.4f}            ║')
print(f'║  ROC-AUC (Macro-avg)   : {macro_roc:.4f}            ║')
print(f'║  Cross-Val Mean (5-Fold): {cv_scores.mean():.4f}           ║')
print('╚══════════════════════════════════════════╝')

---
## 7. 💾 Simpan Model (.pkl)

In [ ]:
# ── Simpan model
MODEL_PATH = 'smartphone_model.pkl'
joblib.dump(rf_model, MODEL_PATH)
print(f'✅ Model disimpan ke: {MODEL_PATH}')

# ── Simpan nama kolom fitur (penting untuk app.py)
feature_names = list(X.columns)
joblib.dump(feature_names, 'feature_names.pkl')
print(f'✅ Nama fitur disimpan ke: feature_names.pkl')

# ── Verifikasi load ulang
model_loaded = joblib.load(MODEL_PATH)
test_pred    = model_loaded.predict(X_test)
test_acc     = accuracy_score(y_test, test_pred)
print(f'\n✅ Verifikasi model loaded: Accuracy = {test_acc:.4f}')

In [ ]:
# ── (Opsional) Download file di Google Colab
# from google.colab import files
# files.download('smartphone_model.pkl')
# files.download('feature_names.pkl')
print('Uncomment baris di atas jika ingin download file dari Colab.')

---
## ✅ Notebook Selesai!

| File Output | Keterangan |
|---|---|
| `smartphone_model.pkl` | Model Random Forest yang sudah dilatih |
| `feature_names.pkl` | Nama kolom fitur (dipakai app.py) |
| `plot_distribusi_target.png` | Grafik distribusi kelas harga |
| `plot_distribusi_fitur.png` | Histogram semua fitur |
| `plot_boxplot_fitur.png` | Boxplot fitur vs kelas |
| `plot_heatmap_korelasi.png` | Heatmap korelasi |
| `plot_confusion_matrix.png` | Confusion matrix |
| `plot_roc_auc.png` | ROC-AUC curve |
| `plot_feature_importance.png` | Feature importance |

**Langkah berikutnya:** Buat `app.py` Streamlit menggunakan `smartphone_model.pkl` di atas.